In [ ]:
import hashlib
import time

class Block:
    def __init__(self, index, transactions, previous_hash=""):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions  # Πλέον είναι λίστα από transactions
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # Ενώνουμε όλα τα transactions σε ένα string
        tx_string = "".join([str(tx) for tx in self.transactions])

        block_string = str(self.index) + str(self.timestamp) + tx_string + self.previous_hash + str(self.nonce)
        return hashlib.sha256(block_string.encode('utf-8')).hexdigest()

In [ ]:
tmp_block = Block(0, ["tx1", "tx2"], "")
tmp_hash = tmp_block.calculate_hash()
print(tmp_hash)

In [ ]:
class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, ["Genesis Block"], "0")

    def add_block(self, new_data):
        new_idx = len(self.chain)

        previous_block = self.get_latest_block()
        previous_hash = previous_block.calculate_hash()

        self.chain.append(Block(new_idx, new_data, previous_hash))

    def get_latest_block(self):
        return self.chain[-1]

In [ ]:
tmp_blockchain = Blockchain()
latest_block = tmp_blockchain.get_latest_block()
print(latest_block.calculate_hash())

In [ ]:
tmp_blockchain.add_block("new transaction")
for i, block in enumerate(tmp_blockchain.chain):
  print(f"Block {i} hash: {block.calculate_hash()}")

In [ ]:
import hashlib
import time

class Block:
    def __init__(self, index, transactions, previous_hash=""):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        tx_string = "".join([str(tx) for tx in self.transactions])
        block_string = (
            str(self.index) +
            str(self.timestamp) +
            tx_string +
            self.previous_hash +
            str(self.nonce)
        )
        return hashlib.sha256(block_string.encode("utf-8")).hexdigest()

In [ ]:
tmp_block = Block(
    0,
     [{"sender": "Alice", "receiver": "Bob", "amount": 10},
      {"sender": "Bob", "receiver": "Charlie", "amount": 5}
     ],
     "0")
tmp_hash = tmp_block.calculate_hash()
print(tmp_hash)

In [ ]:
my_blockchain = Blockchain()

transactions_block_1 = [
    {"sender": "Alice", "receiver": "Bob", "amount": 10},
    {"sender": "Bob", "receiver": "Charlie", "amount": 5}
]

transactions_block_2 = [
    {"sender": "Charlie", "receiver": "Alice", "amount": 2},
    {"sender": "Alice", "receiver": "David", "amount": 7}
]

my_blockchain.add_block(transactions_block_1)
my_blockchain.add_block(transactions_block_2)

for block in my_blockchain.chain:
    print("Index:", block.index)
    print("Timestamp:", block.timestamp)
    print("Transactions:", block.transactions)
    print("Previous Hash:", block.previous_hash)
    print("Hash:", block.hash)
    print("-" * 50)

In [ ]:
# !pip install "web3[tester]"

In [ ]:
from web3 import Web3

# 1. Ξεκινάμε το τοπικό (in-memory) Blockchain.
w3 = Web3(Web3.EthereumTesterProvider())

print(f"Συνδεθήκαμε επιτυχώς; {w3.is_connected()}\n")

# Η βιβλιοθήκη δημιουργεί αυτόματα 10 λογαριασμούς (διευθύνσεις)
# και τους γεμίζει με 1.000.000 εικονικά ETH τον καθένα.
alice = w3.eth.accounts[0]
bob = w3.eth.accounts[1]

print(f"Διεύθυνση Αλίκης: {alice}")
print(f"Διεύθυνση Μπομπ:  {bob}\n")

Στο Blockchain δεν υπάρχουν δεκαδικοί αριθμοί (floats), γιατί οι υπολογιστές κάνουν λάθη στρογγυλοποίησης. Όλα αποθηκεύονται στη μικρότερη δυνατή μονάδα, το Wei. ($1 \text{ ETH} = 10^{18} \text{ Wei}$).

In [ ]:
# 2. Ελέγχουμε τα υπόλοιπα
balance_alice_wei = w3.eth.get_balance(alice)
balance_bob_wei = w3.eth.get_balance(bob)

# Μετατροπή από Wei σε Ether για να μπορούμε να το διαβάσουμε
balance_alice_eth = w3.from_wei(balance_alice_wei, 'ether')
balance_bob_eth = w3.from_wei(balance_bob_wei, 'ether')

print(f"Αρχικό Υπόλοιπο Αλίκης: {balance_alice_eth} ETH")
print(f"Αρχικό Υπόλοιπο Μπομπ:  {balance_bob_eth} ETH\n")

Η Αλίκη θέλει να στείλει 5 ETH στον Μπομπ. Στον προσομοιωτή, δεν χρειάζεται να υπογράψουμε χειροκίνητα τη συναλλαγή με το ιδιωτικό κλειδί (το κάνει αυτόματα ο EthereumTesterProvider για τους λογαριασμούς του), οπότε επικεντρωνόμαστε στη δομή της συναλλαγής.

In [ ]:
print("--- Η Αλίκη στέλνει 5 ETH στον Μπομπ... ---")

# 3. Δημιουργία και αποστολή της συναλλαγής
tx_hash = w3.eth.send_transaction({
    'from': alice,
    'to': bob,
    'value': w3.to_wei(5, 'ether'), # Πρέπει να το στείλουμε σε Wei!
    'gas': 21000 # Το "καύσιμο" που πληρώνουμε στους miners
})

# Περιμένουμε το δίκτυο να βάλει τη συναλλαγή σε ένα Block
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

print(f"Επιτυχία! Η συναλλαγή μπήκε στο Block νούμερο: {receipt.blockNumber}")
print(f"Το Hash της συναλλαγής είναι: {tx_hash.hex()}\n")

In [ ]:
# 4. Ελέγχουμε τα νέα υπόλοιπα
new_alice_eth = w3.from_wei(w3.eth.get_balance(alice), 'ether')
new_bob_eth = w3.from_wei(w3.eth.get_balance(bob), 'ether')

print(f"Νέο Υπόλοιπο Αλίκης: {new_alice_eth} ETH")
print(f"Νέο Υπόλοιπο Μπομπ:  {new_bob_eth} ETH")

Ο Μπομπ είχε 1.000.000 και τώρα έχει 1.000.005. Σωστά. Η Αλίκη είχε 1.000.000, έδωσε 5, άρα θα έπρεπε να έχει 999.995. Αν κοιτάξετε όμως την οθόνη σας, η Αλίκη έχει 999994.999979 ETH. Πού πήγαν τα υπόλοιπα χρήματα;